# nuReasoning Dataset Viewer

Jupyter notebook for viewing and understanding nuReasoning clips.

Each clip folder is expected to contain `metadata.json`. Frame metadata points to camera images, ego-state pickles, annotation pickles, optional lidar point clouds, and optional map annotations. Use this notebook to inspect clip structure and render camera/BEV views as PNG frames or MP4 videos.

Toggle `RENDER_LIDAR` to add or remove the lidar panel.

In [1]:
# Configuration
# Set INPUT_ROOT to a folder containing clip directories, or set CLIP_PATH to one clip folder.
INPUT_ROOT = "./data"
CLIP_PATH = "/home/etri/DATASET/nureasoning/clips/2023.07.31.23.53.58_KMHKM4AEXM1P98033_169ca87cd5ca5fdb912f770aa679edd7"  # Set to a clip folder to inspect one clip.
OUTPUT_DIR = "./nureasoning_viz"

# Frame/clip selection
START_INDEX = 0
END_INDEX = -1
STRIDE = 1
MAX_FRAMES = 0  # 0 = all selected frames
MAX_CLIPS = 0  # 0 = all discovered clips
FRAME_IDX = None  # Set to an int to render one PNG frame instead of an MP4

# Render options
FPS = 10
DPI = 150
BEV_RANGE_M = 80.0
RENDER_LIDAR = True  # Set True to include the lidar panel when rendering.
LIDAR_RANGE_M = 80.0
MAX_LIDAR_POINTS = 50_000
TRAJECTORY_DT_S = 0.1
SHOW_OBJECTS = True
FILTER_TO_CLIPS_WITH_LIDAR = False
SKIP_EXISTING = True

## Install Packages

Run this cell if the imports below are missing in your notebook environment.

In [2]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "matplotlib": "matplotlib",
    "numpy": "numpy",
    "shapely": "shapely",
}

missing_packages = [package for module, package in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
else:
    print("All required packages are already installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 38.0 MB/s  0:00:00


## Data Model And Rendering Code

Run this cell to define the data loading, visualization, PCD parsing, and video rendering helpers used below.

In [3]:
from __future__ import annotations

import gc
import importlib
import json
import math
import os
import pickle
import shutil
import struct
import subprocess
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib
import matplotlib.image as mpimg
import matplotlib.patches as mpatches
import numpy as np
from matplotlib.animation import FFMpegWriter
from matplotlib.lines import Line2D
from shapely.geometry import LineString

# Matplotlib backend is managed by Jupyter.
import matplotlib.pyplot as plt

from data_schema import Annotations, EgoState, nuReasoningStaticMap

VEHICLE_REAR_LENGTH = 0.79

LIGHT_GREY = "#D3D3D3"
NEW_TAB_10: Dict[int, str] = {
    0: "#4e79a7",
    1: "#f28e2b",
    2: "#e15759",
    3: "#76b7b2",
    4: "#59a14f",
    5: "#edc948",
    6: "#b07aa1",
    7: "#ff9da7",
    8: "#9c755f",
    9: "#bab0ac",
}
ELLIS_5: Dict[int, str] = {
    0: "#DE7061",
    1: "#B0E685",
    2: "#4AC4BD",
    3: "#E38C47",
    4: "#699CDB",
}

FIG_MARGIN = 0.006
GC_COLLECT_EVERY_FRAMES = 25

CAMERA_LAYOUT: List[List[Optional[str]]] = [
    ["front_left", "front", "front_right"],
    ["left", None, "right"],
    ["back_left", "back", "back_right"],
]
CAMERA_ORDER = [cam for row in CAMERA_LAYOUT for cam in row if cam is not None]


@dataclass(frozen=True)
class PlotStyle:
    fill_color: Optional[str] = None
    fill_alpha: float = 0.0
    line_color: Optional[str] = None
    line_alpha: float = 1.0
    line_width: float = 1.0
    line_style: str = "-"
    marker: Optional[str] = None
    marker_size: float = 0.0
    marker_edge_color: Optional[str] = None
    zorder: int = 1


@dataclass
class ClipContext:
    clip_path: str
    clip_name: str
    metadata: Dict[str, Any]
    frames: List[Dict[str, Any]]


MAP_LAYER_STYLES: Dict[str, PlotStyle] = {
    "lane": PlotStyle(fill_color=LIGHT_GREY, fill_alpha=1.0, line_color=LIGHT_GREY, line_alpha=0.0, zorder=1),
    "road_block": PlotStyle(fill_color="#0000C0", fill_alpha=0.10, line_color="#0000C0", line_alpha=0.35, zorder=0),
    "intersection": PlotStyle(fill_color=LIGHT_GREY, fill_alpha=1.0, line_color=LIGHT_GREY, line_alpha=0.0, zorder=0),
    "crosswalk": PlotStyle(fill_color=NEW_TAB_10[6], fill_alpha=0.28, line_color=NEW_TAB_10[6], line_alpha=0.0, zorder=1),
    "stop_polygon": PlotStyle(fill_color="#FF0101", fill_alpha=0.18, line_color="#FF0101", line_alpha=0.25, zorder=2),
    "lane_centerline": PlotStyle(line_color="#666666", line_alpha=0.50, line_width=0.9, line_style="-", zorder=2),
    "lane_connector": PlotStyle(line_color="#CBCBCB", line_alpha=0.95, line_width=1.0, line_style="-", zorder=2),
    "baseline_path": PlotStyle(line_color="#666666", line_alpha=0.9, line_width=1.0, line_style="--", zorder=2),
}

ACTOR_STYLES: Dict[str, PlotStyle] = {
    "vehicle": PlotStyle(fill_color=ELLIS_5[4], fill_alpha=1.0, line_color="black", line_alpha=1.0, zorder=6),
    "pedestrian": PlotStyle(fill_color=NEW_TAB_10[6], fill_alpha=1.0, line_color="black", line_alpha=1.0, zorder=6),
    "bicycle": PlotStyle(fill_color=ELLIS_5[3], fill_alpha=1.0, line_color="black", line_alpha=1.0, zorder=6),
    "generic": PlotStyle(fill_color=NEW_TAB_10[5], fill_alpha=1.0, line_color="black", line_alpha=1.0, zorder=6),
    "ego": PlotStyle(fill_color=ELLIS_5[0], fill_alpha=1.0, line_color="black", line_alpha=1.0, line_width=1.4, zorder=8),
}

TRAJECTORY_STYLES: Dict[str, PlotStyle] = {
    "history": PlotStyle(line_color="#2c3e50", line_alpha=0.85, line_width=2.0, marker="o", marker_size=3.5, zorder=7),
    "future": PlotStyle(line_color=NEW_TAB_10[4], line_alpha=1.0, line_width=2.0, marker="o", marker_size=4.0, marker_edge_color="black", zorder=7),
    "route": PlotStyle(line_color="#8e44ad", line_alpha=0.75, line_width=2.0, line_style="-", zorder=3),
}

SHOW_LIDAR_PANEL = bool(globals().get("RENDER_LIDAR", False))

TL_STATE_COLORS: Dict[str, str] = {
    "red": "#e74c3c",
    "yellow": "#f1c40f",
    "green": "#2ecc71",
    "off": "#95a5a6",
    "unknown": "#7f8c8d",
}


def _install_pickle_aliases() -> None:
    alias_map = {"data_schema_v0": "data_schema"}
    for old_name, new_name in alias_map.items():
        try:
            module = importlib.import_module(new_name)
            import sys

            sys.modules.setdefault(old_name, module)
        except Exception:
            pass


def _load_pickle(path: str) -> Any:
    _install_pickle_aliases()
    with open(path, "rb") as f:
        return pickle.load(f)


def _load_json(path: str) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        loaded = json.load(f)
    return loaded if isinstance(loaded, dict) else {}


def _resolve_path(path_value: str, clip_path: str) -> str:
    if not path_value:
        return ""
    expanded = os.path.expanduser(str(path_value))
    if os.path.isabs(expanded):
        return expanded
    by_clip = os.path.abspath(os.path.join(clip_path, expanded))
    if os.path.exists(by_clip):
        return by_clip
    direct = os.path.abspath(expanded)
    if os.path.exists(direct):
        return direct
    return by_clip


def _discover_clips(
    input_root: str,
    clip_path: str,
    require_lidar: bool,
    max_clips: int,
) -> List[ClipContext]:
    if clip_path:
        clip_abs = os.path.abspath(clip_path)
        return [_build_clip_context(clip_abs)]

    root_abs = os.path.abspath(input_root)
    if not os.path.isdir(root_abs):
        raise FileNotFoundError(f"Input root not found: {root_abs}")

    clips: List[ClipContext] = []
    for name in sorted(os.listdir(root_abs)):
        candidate = os.path.join(root_abs, name)
        if not os.path.isdir(candidate):
            continue
        metadata_path = os.path.join(candidate, "metadata.json")
        if not os.path.isfile(metadata_path):
            continue
        clip = _build_clip_context(candidate)
        if require_lidar and not any(_frame_lidar_path(frame, clip.clip_path) for frame in clip.frames):
            continue
        clips.append(clip)
        if max_clips > 0 and len(clips) >= max_clips:
            break
    return clips


def _build_clip_context(clip_path: str) -> ClipContext:
    metadata_path = os.path.join(clip_path, "metadata.json")
    if not os.path.isfile(metadata_path):
        raise FileNotFoundError(f"metadata.json not found in clip path: {clip_path}")
    metadata = _load_json(metadata_path)
    frames = metadata.get("frames", [])
    if not isinstance(frames, list):
        frames = []
    return ClipContext(
        clip_path=clip_path,
        clip_name=os.path.basename(clip_path),
        metadata=metadata,
        frames=[frame for frame in frames if isinstance(frame, dict)],
    )


def _frame_lidar_path(frame: Dict[str, Any], clip_path: str) -> str:
    sensors = frame.get("sensors", {})
    lidar = sensors.get("lidar", {}) if isinstance(sensors, dict) else {}
    rel_path = lidar.get("point_cloud_path", "") if isinstance(lidar, dict) else ""
    path = _resolve_path(str(rel_path), clip_path)
    return path if path and os.path.isfile(path) else ""


def _safe_image(path: str) -> Optional[np.ndarray]:
    if not path or not os.path.isfile(path):
        return None
    try:
        return mpimg.imread(path)
    except Exception:
        return None


def _get_field(obj: Any, key: str, default: Any = None) -> Any:
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)


def _safe_float(value: Any, default: float = 0.0) -> float:
    try:
        return float(value)
    except Exception:
        return default


def _box_corners_2d(cx: float, cy: float, half_l: float, half_w: float, yaw: float) -> List[Tuple[float, float]]:
    c, s = math.cos(yaw), math.sin(yaw)
    dx = [half_l, half_l, -half_l, -half_l]
    dy = [half_w, -half_w, -half_w, half_w]
    return [(cx + c * dx[i] - s * dy[i], cy + s * dx[i] + c * dy[i]) for i in range(4)]


def _ego_box_center_from_base_link(
    x: float,
    y: float,
    yaw: float,
    ego_length: float,
    vehicle_rear_length: float = VEHICLE_REAR_LENGTH,
) -> Tuple[float, float]:
    center_offset = ego_length / 2.0 - vehicle_rear_length
    return x + math.cos(yaw) * center_offset, y + math.sin(yaw) * center_offset


def _build_raw_route_xy(route_path: Any) -> Optional[np.ndarray]:
    pts: List[Tuple[float, float]] = []
    for p in route_path:
        if isinstance(p, (list, tuple, np.ndarray)) and len(p) >= 2:
            pts.append((float(p[0]), float(p[1])))
    if len(pts) < 2:
        return None
    return np.asarray(pts, dtype=np.float64)


def build_route_line(mission_goal: Any) -> Optional[LineString]:
    if mission_goal is None:
        return None
    route_path = mission_goal.get("route_path") if isinstance(mission_goal, dict) else getattr(mission_goal, "route_path", None)
    if not route_path or len(route_path) < 2:
        return None
    route_xy = _build_raw_route_xy(route_path)
    if route_xy is None or len(route_xy) < 2:
        return None
    return LineString(route_xy)


def _yaw_from_pose(pose: Dict[str, Any]) -> float:
    if "yaw" in pose:
        return _safe_float(pose.get("yaw"), 0.0)
    qw = _safe_float(pose.get("qw"), 1.0)
    qx = _safe_float(pose.get("qx"), 0.0)
    qy = _safe_float(pose.get("qy"), 0.0)
    qz = _safe_float(pose.get("qz"), 0.0)
    siny_cosp = 2.0 * (qw * qz + qx * qy)
    cosy_cosp = 1.0 - 2.0 * (qy * qy + qz * qz)
    return float(np.arctan2(siny_cosp, cosy_cosp))


def _viz_configure_ax(ax: Any) -> None:
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)


def _viz_configure_bev_ax(ax: Any, ego_x: float, ego_y: float, margin_m: float) -> None:
    ax.set_aspect("equal")
    ax.set_xlim(ego_x - margin_m, ego_x + margin_m)
    ax.set_ylim(ego_y - margin_m, ego_y + margin_m)
    ax.set_facecolor("white")
    _viz_configure_ax(ax)


def _viz_geometry_xy(geometry: Any, min_points: int = 2) -> Optional[np.ndarray]:
    if not geometry:
        return None
    pts = np.asarray(geometry, dtype=np.float64)
    if pts.ndim != 2 or pts.shape[0] < min_points or pts.shape[1] < 2:
        return None
    return pts[:, :2]


def _viz_geometry_center(geometry: Any) -> Optional[np.ndarray]:
    pts = _viz_geometry_xy(geometry, min_points=1)
    if pts is None:
        return None
    return pts.mean(axis=0)


def _viz_add_polygon(ax: Any, geometry: Any, style: PlotStyle) -> None:
    pts = _viz_geometry_xy(geometry, min_points=3)
    if pts is None:
        return
    patch = mpatches.Polygon(
        pts,
        closed=True,
        facecolor=style.fill_color or "none",
        edgecolor=style.line_color or style.fill_color or "none",
        alpha=style.fill_alpha if style.fill_color else style.line_alpha,
        linewidth=style.line_width,
        linestyle=style.line_style,
        zorder=style.zorder,
    )
    ax.add_patch(patch)


def _viz_add_polyline(ax: Any, geometry: Any, style: PlotStyle) -> None:
    pts = _viz_geometry_xy(geometry, min_points=2)
    if pts is None:
        return
    ax.plot(
        pts[:, 0],
        pts[:, 1],
        color=style.line_color,
        alpha=style.line_alpha,
        linewidth=style.line_width,
        linestyle=style.line_style,
        zorder=style.zorder,
    )


def _viz_add_box(
    ax: Any,
    cx: float,
    cy: float,
    length: float,
    width: float,
    yaw: float,
    style: PlotStyle,
    add_heading: bool = True,
) -> None:
    if length <= 0 or width <= 0:
        return
    corners = np.asarray(_box_corners_2d(cx, cy, length / 2.0, width / 2.0, yaw), dtype=np.float64)
    closed = np.vstack([corners, corners[0]])
    if style.fill_color:
        ax.fill(closed[:, 0], closed[:, 1], color=style.fill_color, alpha=style.fill_alpha, zorder=style.zorder)
    if style.line_color:
        ax.plot(
            closed[:, 0],
            closed[:, 1],
            color=style.line_color,
            alpha=style.line_alpha,
            linewidth=style.line_width,
            linestyle=style.line_style,
            zorder=style.zorder,
        )
    if add_heading and style.line_color:
        heading_len = max(1.0, length * 0.35)
        heading = np.array([[cx, cy], [cx + math.cos(yaw) * heading_len, cy + math.sin(yaw) * heading_len]], dtype=np.float64)
        ax.plot(
            heading[:, 0],
            heading[:, 1],
            color=style.line_color,
            alpha=style.line_alpha,
            linewidth=style.line_width,
            linestyle=style.line_style,
            zorder=style.zorder + 0.1,
        )


def _viz_sample_trajectory_for_plot(trajectory: np.ndarray, trajectory_dt_s: float, plot_dt_s: float = 0.5) -> np.ndarray:
    if len(trajectory) <= 1:
        return trajectory
    dt = max(float(trajectory_dt_s), 1e-6)
    stride = max(1, int(round(float(plot_dt_s) / dt)))
    sampled = trajectory[::stride]
    if not np.array_equal(sampled[-1], trajectory[-1]):
        sampled = np.vstack([sampled, trajectory[-1]])
    return sampled


def _viz_add_trajectory(
    ax: Any,
    trajectory: Optional[Any],
    style: PlotStyle,
    *,
    trajectory_dt_s: float = 0.1,
    plot_dt_s: float = 0.5,
) -> None:
    if trajectory is None:
        return
    pts = np.asarray(trajectory, dtype=np.float64)
    if pts.ndim != 2 or pts.shape[0] == 0 or pts.shape[1] < 2:
        return
    pts = _viz_sample_trajectory_for_plot(pts[:, :2], trajectory_dt_s, plot_dt_s)
    ax.plot(
        pts[:, 0],
        pts[:, 1],
        color=style.line_color,
        alpha=style.line_alpha,
        linewidth=style.line_width,
        linestyle=style.line_style,
        marker=style.marker,
        markersize=style.marker_size,
        markeredgecolor=style.marker_edge_color,
        zorder=style.zorder,
    )


def _viz_classify_actor(category: str) -> str:
    cat = (category or "").lower()
    if "pedestrian" in cat or cat == "human":
        return "pedestrian"
    if "bike" in cat or "cycle" in cat:
        return "bicycle"
    if "vehicle" in cat or "car" in cat or "truck" in cat or "bus" in cat:
        return "vehicle"
    return "generic"


def _viz_build_map_lookup(static_map: nuReasoningStaticMap) -> Dict[str, Dict[int, Any]]:
    return {
        "lane_connectors": {int(obj.id): obj for obj in getattr(static_map, "lane_connectors", []) if getattr(obj, "id", None) is not None},
        "road_blocks": {int(obj.id): obj for obj in getattr(static_map, "road_blocks", []) if getattr(obj, "id", None) is not None},
        "traffic_lights": {int(obj.id): obj for obj in getattr(static_map, "traffic_lights", []) if getattr(obj, "id", None) is not None},
    }


def _viz_resolve_tl_center(tl_state: Any, static_map_lookup: Dict[str, Dict[int, Any]]) -> Optional[np.ndarray]:
    lane_connector_id = getattr(tl_state, "lane_connector_id", None)
    if lane_connector_id is not None:
        lane_connector = static_map_lookup["lane_connectors"].get(int(lane_connector_id))
        if lane_connector is not None:
            center = _viz_geometry_center(getattr(lane_connector, "geometry", None))
            if center is not None:
                return center

    roadblock_id = getattr(tl_state, "roadblock_id", None)
    if roadblock_id is not None:
        road_block = static_map_lookup["road_blocks"].get(int(roadblock_id))
        if road_block is not None:
            center = _viz_geometry_center(getattr(road_block, "geometry", None))
            if center is not None:
                return center

    tl_id = getattr(tl_state, "id", None)
    if tl_id is not None:
        traffic_light = static_map_lookup["traffic_lights"].get(int(tl_id))
        if traffic_light is not None:
            return _viz_geometry_center(getattr(traffic_light, "geometry", None))

    return None


def _viz_draw_static_map(ax: Any, static_map: nuReasoningStaticMap) -> Dict[str, Dict[int, Any]]:
    for road_block in getattr(static_map, "road_blocks", []):
        _viz_add_polygon(ax, road_block.geometry, MAP_LAYER_STYLES["road_block"])
    for intersection in getattr(static_map, "intersections", []):
        _viz_add_polygon(ax, intersection.geometry, MAP_LAYER_STYLES["intersection"])
    for lane in getattr(static_map, "lanes", []):
        _viz_add_polygon(ax, lane.polygon, MAP_LAYER_STYLES["lane"])
        _viz_add_polyline(ax, getattr(lane, "centerline", None), MAP_LAYER_STYLES["lane_centerline"])
    for crosswalk in getattr(static_map, "crosswalks", []):
        _viz_add_polygon(ax, crosswalk.geometry, MAP_LAYER_STYLES["crosswalk"])
    for stop_polygon in getattr(static_map, "stop_polygons", []):
        _viz_add_polygon(ax, stop_polygon.geometry, MAP_LAYER_STYLES["stop_polygon"])
    for lane_connector in getattr(static_map, "lane_connectors", []):
        _viz_add_polyline(ax, lane_connector.geometry, MAP_LAYER_STYLES["lane_connector"])
    for baseline_path in getattr(static_map, "baseline_paths", []):
        _viz_add_polyline(ax, baseline_path.geometry, MAP_LAYER_STYLES["baseline_path"])
    return _viz_build_map_lookup(static_map)


def _viz_draw_traffic_light_states(
    ax: Any,
    ann: Annotations,
    static_map_lookup: Optional[Dict[str, Dict[int, Any]]],
) -> None:
    if static_map_lookup is None:
        return
    for tl_state in ann.traffic_light_states:
        center = _viz_resolve_tl_center(tl_state, static_map_lookup)
        if center is None:
            continue
        state = (getattr(tl_state, "state", None) or "unknown").lower()
        color = TL_STATE_COLORS.get(state, TL_STATE_COLORS["unknown"])
        ax.scatter([center[0]], [center[1]], s=42, c=color, edgecolors="black", linewidths=0.6, zorder=7)


def _viz_draw_objects(ax: Any, ann: Annotations, show_objects: bool) -> None:
    if not show_objects:
        return
    for obj in ann.objects:
        cat = (obj.category or "").strip().lower()
        actor_key = _viz_classify_actor(cat)
        _viz_add_box(
            ax,
            cx=float(obj.pose.get("x", 0.0)),
            cy=float(obj.pose.get("y", 0.0)),
            length=float(obj.dimensions.get("l", 0.0)),
            width=float(obj.dimensions.get("w", 0.0)),
            yaw=float(obj.pose.get("yaw", 0.0)),
            style=ACTOR_STYLES[actor_key],
            add_heading=True,
        )


def _metadata_ego_dimensions(metadata: Dict[str, Any]) -> Dict[str, float]:
    raw_dims = metadata.get("ego_dimensions", {})
    if not isinstance(raw_dims, dict):
        return {}

    dims: Dict[str, float] = {}
    key_map = {
        "length": "l",
        "width": "w",
        "height": "h",
        "vehicle_rear_length": "vehicle_rear_length",
        "l": "l",
        "w": "w",
        "h": "h",
    }
    for raw_key, normalized_key in key_map.items():
        if raw_key in raw_dims:
            dims[normalized_key] = _safe_float(raw_dims[raw_key], 0.0)
    return dims


def _viz_draw_camera_center_ego(
    ax: Any,
    ego_state: EgoState,
) -> None:
    ax.clear()
    ax.set_facecolor("black")
    ax.axis("off")

    pose = getattr(ego_state, "pose", {}) or {}
    velocity = getattr(ego_state, "velocity", {}) or {}
    acceleration = getattr(ego_state, "acceleration", {}) or {}
    x = _safe_float(pose.get("x", 0.0))
    y = _safe_float(pose.get("y", 0.0))
    z = _safe_float(pose.get("z", 0.0))
    yaw = _yaw_from_pose(pose)
    vx = _safe_float(velocity.get("x", velocity.get("vx", 0.0)))
    vy = _safe_float(velocity.get("y", velocity.get("vy", 0.0)))
    vz = _safe_float(velocity.get("z", velocity.get("vz", 0.0)))
    ax_val = _safe_float(acceleration.get("x", acceleration.get("ax", 0.0)))
    ay_val = _safe_float(acceleration.get("y", acceleration.get("ay", 0.0)))
    az_val = _safe_float(acceleration.get("z", acceleration.get("az", 0.0)))
    speed = math.sqrt(vx * vx + vy * vy + vz * vz)
    accel = math.sqrt(ax_val * ax_val + ay_val * ay_val + az_val * az_val)
    card = mpatches.FancyBboxPatch(
        (0.06, 0.08),
        0.88,
        0.84,
        boxstyle="round,pad=0.025,rounding_size=0.04",
        transform=ax.transAxes,
        facecolor="#111827",
        edgecolor="#6b7280",
        linewidth=1.0,
        alpha=0.96,
    )
    ax.add_patch(card)
    ax.text(
        0.5,
        0.84,
        "EGO STATE",
        transform=ax.transAxes,
        color="#93c5fd",
        fontsize=8.5,
        fontweight="bold",
        ha="center",
        va="center",
    )
    ax.text(
        0.5,
        0.70,
        f"{speed:.2f} m/s",
        transform=ax.transAxes,
        color="white",
        fontsize=13,
        fontweight="bold",
        ha="center",
        va="center",
    )
    ax.text(
        0.5,
        0.60,
        f"accel {accel:.2f} m/s^2",
        transform=ax.transAxes,
        color="#d1d5db",
        fontsize=7.5,
        ha="center",
        va="center",
    )
    state_lines = [
        f"pose   x {x:7.2f}   y {y:7.2f}   z {z:5.2f}",
        f"yaw    {math.degrees(yaw):7.1f} deg",
        f"vel    x {vx:7.2f}   y {vy:7.2f}   z {vz:5.2f}",
        f"acc    x {ax_val:7.2f}   y {ay_val:7.2f}   z {az_val:5.2f}",
    ]
    ax.text(
        0.10,
        0.45,
        "\n".join(state_lines),
        transform=ax.transAxes,
        color="#f9fafb",
        fontsize=6.5,
        family="monospace",
        ha="left",
        va="top",
        linespacing=1.35,
    )


def _viz_draw_ego(ax: Any, ego_state: EgoState, metadata_ego_dims: Optional[Dict[str, float]] = None) -> None:
    ego_pose = ego_state.pose
    ego_dims = metadata_ego_dims or {}
    if not ego_dims:
        ego_dims = getattr(ego_state, "dimensions", None) or {}
    length = float(ego_dims.get("l", 5.176))
    yaw = _yaw_from_pose(ego_pose)
    ego_cx, ego_cy = _ego_box_center_from_base_link(
        float(ego_pose.get("x", 0.0)),
        float(ego_pose.get("y", 0.0)),
        yaw,
        length,
        float(ego_dims.get("vehicle_rear_length", VEHICLE_REAR_LENGTH)),
    )
    _viz_add_box(
        ax,
        cx=ego_cx,
        cy=ego_cy,
        length=length,
        width=float(ego_dims.get("w", 2.297)),
        yaw=yaw,
        style=ACTOR_STYLES["ego"],
        add_heading=True,
    )


def _viz_add_legend(ax: Any) -> None:
    handles = [
        mpatches.Patch(facecolor=MAP_LAYER_STYLES["lane"].fill_color, edgecolor="none", label="Lane"),
        mpatches.Patch(facecolor=MAP_LAYER_STYLES["crosswalk"].fill_color, edgecolor="none", label="Crosswalk"),
        Line2D([0], [0], color=MAP_LAYER_STYLES["baseline_path"].line_color, linestyle="--", linewidth=1.0, label="Baseline"),
        Line2D([0], [0], color=TRAJECTORY_STYLES["route"].line_color, linewidth=2.0, label="Route"),
        mpatches.Patch(facecolor=ACTOR_STYLES["ego"].fill_color, edgecolor="black", label="Ego"),
        mpatches.Patch(facecolor=ACTOR_STYLES["vehicle"].fill_color, edgecolor="black", label="Vehicle"),
        Line2D([0], [0], color=TRAJECTORY_STYLES["history"].line_color, marker="o", linewidth=2.0, label="Ego history"),
        Line2D([0], [0], color=TRAJECTORY_STYLES["future"].line_color, marker="o", linewidth=2.0, label="Ego future"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor=TL_STATE_COLORS["red"], markeredgecolor="black", label="TL state"),
    ]
    ax.legend(handles=handles, loc="upper left", fontsize=7, framealpha=0.92, ncol=2)


def _get_pcd_data_type(pcd_path: str) -> str:
    with open(pcd_path, "rb") as f:
        for _ in range(30):
            line = f.readline()
            if not line:
                break
            if line.strip().lower().startswith(b"data"):
                parts = line.strip().split()
                return parts[1].decode("utf-8").lower() if len(parts) > 1 else "binary"
    return "binary"


def _read_ascii_pcd_points(path: str) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    header_lines = 0
    fields: List[str] = []
    with open(path, "rb") as f:
        for raw_line in f:
            header_lines += 1
            line = raw_line.decode("utf-8", errors="ignore").strip()
            lower = line.lower()
            if lower.startswith("fields"):
                fields = line.split()[1:]
            if lower.startswith("data"):
                break
    data = np.loadtxt(path, skiprows=header_lines)
    if data.ndim == 1:
        data = data.reshape(1, -1)
    xyz = data[:, :3].T
    intensity = None
    if "intensity" in fields:
        intensity = data[:, fields.index("intensity")]
    elif data.shape[1] > 3:
        intensity = data[:, 3]
    return xyz, intensity


def _parse_pcd_header(path: str) -> Tuple[Dict[str, Any], int]:
    header: Dict[str, Any] = {}
    header_size = 0
    with open(path, "rb") as f:
        for raw_line in f:
            header_size += len(raw_line)
            line = raw_line.decode("utf-8", errors="ignore").strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            key = parts[0].lower()
            values = parts[1:]
            header[key] = values
            if key == "data":
                break
    return header, header_size


def _pcd_numpy_dtype(header: Dict[str, Any]) -> np.dtype:
    fields = header.get("fields", [])
    sizes = [int(v) for v in header.get("size", [])]
    types = header.get("type", [])
    counts = [int(v) for v in header.get("count", ["1"] * len(fields))]
    if not (len(fields) == len(sizes) == len(types) == len(counts)):
        raise ValueError("Malformed PCD header: FIELDS/SIZE/TYPE/COUNT length mismatch")

    dtype_fields: List[Tuple[str, Any]] = []
    for name, size, type_code, count in zip(fields, sizes, types, counts):
        key = (type_code.upper(), size)
        if key == ("F", 4):
            base = "<f4"
        elif key == ("F", 8):
            base = "<f8"
        elif key == ("U", 1):
            base = "u1"
        elif key == ("U", 2):
            base = "<u2"
        elif key == ("U", 4):
            base = "<u4"
        elif key == ("I", 1):
            base = "i1"
        elif key == ("I", 2):
            base = "<i2"
        elif key == ("I", 4):
            base = "<i4"
        else:
            raise ValueError(f"Unsupported PCD field type {type_code} size {size}")
        dtype_fields.append((name, base) if count == 1 else (name, base, (count,)))
    return np.dtype(dtype_fields)


def _lzf_decompress(data: bytes, expected_size: int) -> bytes:
    output = bytearray()
    i = 0
    data_len = len(data)
    while i < data_len:
        ctrl = data[i]
        i += 1
        if ctrl < 32:
            length = ctrl + 1
            output.extend(data[i : i + length])
            i += length
        else:
            length = ctrl >> 5
            ref_offset = (ctrl & 0x1F) << 8
            if length == 7:
                if i >= data_len:
                    raise ValueError("Truncated LZF length extension")
                length += data[i]
                i += 1
            if i >= data_len:
                raise ValueError("Truncated LZF back reference")
            ref_offset += data[i]
            i += 1
            length += 2
            ref_pos = len(output) - ref_offset - 1
            if ref_pos < 0:
                raise ValueError("Invalid LZF back reference")
            for _ in range(length):
                output.append(output[ref_pos])
                ref_pos += 1
        if len(output) > expected_size:
            raise ValueError("LZF output exceeded expected size")
    if len(output) != expected_size:
        raise ValueError(f"LZF output size {len(output)} != expected {expected_size}")
    return bytes(output)


def _read_binary_compressed_pcd_points(path: str) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    header, header_size = _parse_pcd_header(path)
    fields = header.get("fields", [])
    point_count = int(header.get("points", [header.get("width", ["0"])[0]])[0])
    dtype = _pcd_numpy_dtype(header)

    with open(path, "rb") as f:
        f.seek(header_size)
        sizes = f.read(8)
        if len(sizes) != 8:
            raise ValueError("Missing binary_compressed PCD size header")
        compressed_size, uncompressed_size = struct.unpack("<II", sizes)
        compressed = f.read(compressed_size)
    raw = _lzf_decompress(compressed, uncompressed_size)

    # PCD binary_compressed stores structure-of-arrays by field. Convert the first
    # x/y/z/intensity channels without materializing unused lidar metadata fields.
    field_sizes = [int(v) * int(c) for v, c in zip(header.get("size", []), header.get("count", []))]
    offsets = np.cumsum([0] + [size * point_count for size in field_sizes])
    values: Dict[str, np.ndarray] = {}
    for name in ("x", "y", "z", "intensity"):
        if name not in fields:
            continue
        idx = fields.index(name)
        start, end = int(offsets[idx]), int(offsets[idx + 1])
        field_dtype = dtype.fields[name][0]
        values[name] = np.frombuffer(raw[start:end], dtype=field_dtype, count=point_count)

    if not {"x", "y", "z"}.issubset(values):
        raise ValueError("PCD file is missing x/y/z fields")
    xyz = np.vstack([values["x"], values["y"], values["z"]]).astype(np.float32, copy=False)
    intensity = values.get("intensity")
    return xyz, intensity.astype(np.float32, copy=False) if intensity is not None else None


def _read_binary_pcd_points(path: str) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    header, header_size = _parse_pcd_header(path)
    point_count = int(header.get("points", [header.get("width", ["0"])[0]])[0])
    dtype = _pcd_numpy_dtype(header)
    points = np.fromfile(path, dtype=dtype, count=point_count, offset=header_size)
    for name in ("x", "y", "z"):
        if name not in points.dtype.names:
            raise ValueError("PCD file is missing x/y/z fields")
    xyz = np.vstack([points["x"], points["y"], points["z"]]).astype(np.float32, copy=False)
    intensity = points["intensity"].astype(np.float32, copy=False) if "intensity" in points.dtype.names else None
    return xyz, intensity


def load_pcd_points(pcd_path: str) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    data_type = _get_pcd_data_type(pcd_path)
    if data_type == "ascii":
        return _read_ascii_pcd_points(pcd_path)
    if data_type == "binary_compressed":
        return _read_binary_compressed_pcd_points(pcd_path)
    if data_type == "binary":
        return _read_binary_pcd_points(pcd_path)
    raise RuntimeError(f"Unsupported PCD DATA type {data_type!r} in {pcd_path}")


class NuReasoningDataViewer:
    def __init__(
        self,
        clip: ClipContext,
        *,
        bev_range_m: float,
        lidar_range_m: float,
        max_lidar_points: int,
        trajectory_dt_s: float,
        show_objects: bool,
    ):
        if not clip.frames:
            raise ValueError(f"No frames found in clip: {clip.clip_path}")
        self.clip = clip
        self.bev_range_m = bev_range_m
        self.lidar_range_m = lidar_range_m
        self.max_lidar_points = max_lidar_points
        self.trajectory_dt_s = trajectory_dt_s
        self.show_objects = show_objects

        self.static_map = self._load_map()

        figure_size = (28, 8) if SHOW_LIDAR_PANEL else (20, 8)
        self.fig = plt.figure(figsize=figure_size, facecolor="black")
        self.fig.subplots_adjust(
            left=FIG_MARGIN,
            right=1.0 - FIG_MARGIN,
            bottom=FIG_MARGIN,
            top=1.0 - FIG_MARGIN,
            wspace=0,
            hspace=0,
        )
        if SHOW_LIDAR_PANEL:
            outer = self.fig.add_gridspec(1, 3, width_ratios=[1.25, 1.25, 1.06], wspace=0.0)
        else:
            outer = self.fig.add_gridspec(1, 2, width_ratios=[1.25, 1.25], wspace=0.0)
        cam_grid = outer[0].subgridspec(3, 3, hspace=0.0, wspace=0.0)
        self.cam_axes: Dict[str, Any] = {}
        self.center_cam_ax = None
        for row_idx, row in enumerate(CAMERA_LAYOUT):
            for col_idx, cam in enumerate(row):
                ax = self.fig.add_subplot(cam_grid[row_idx, col_idx])
                if cam is None:
                    self.center_cam_ax = ax
                else:
                    self.cam_axes[cam] = ax
        self.bev_ax = self.fig.add_subplot(outer[1])
        self.lidar_ax = self.fig.add_subplot(outer[2]) if SHOW_LIDAR_PANEL else None

    def _load_map(self) -> Optional[nuReasoningStaticMap]:
        map_rel = self.clip.metadata.get("map_annotation", "map.pkl")
        map_path = _resolve_path(str(map_rel), self.clip.clip_path)
        if not os.path.isfile(map_path):
            return None
        try:
            return _load_pickle(map_path)
        except Exception:
            return None

    def _cached_image(self, path: str) -> Optional[np.ndarray]:
        return _safe_image(path)

    def _load_ego_state(self, frame: Dict[str, Any]) -> EgoState:
        path = _resolve_path(str(frame.get("ego_state", "")), self.clip.clip_path)
        return _load_pickle(path)

    def _load_annotations(self, frame: Dict[str, Any]) -> Annotations:
        rel_path = str(frame.get("annotations", "") or "")
        path = _resolve_path(rel_path, self.clip.clip_path)
        if not os.path.isfile(path):
            timestamp_us = frame.get("timestamp_us")
            if timestamp_us is None:
                raise FileNotFoundError(f"annotations path not found in frame metadata: {rel_path!r}")
            path = _resolve_path(f"annotations/{timestamp_us}.pkl", self.clip.clip_path)
        return _load_pickle(path)

    def _draw_cameras(self, frame: Dict[str, Any], ego_state: EgoState) -> None:
        sensors = frame.get("sensors", {})
        cameras = sensors.get("cameras", {}) if isinstance(sensors, dict) else {}
        if self.center_cam_ax is not None:
            _viz_draw_camera_center_ego(
                self.center_cam_ax,
                ego_state,
            )
        for cam in CAMERA_ORDER:
            ax = self.cam_axes[cam]
            ax.clear()
            ax.axis("off")
            rel_path = cameras.get(cam, "") if isinstance(cameras, dict) else ""
            img = self._cached_image(_resolve_path(str(rel_path), self.clip.clip_path))
            if img is None:
                ax.set_facecolor("#111111")
                ax.text(0.5, 0.5, "image missing", color="white", ha="center", va="center", transform=ax.transAxes)
                continue
            ax.imshow(img)
            ax.text(
                0.02,
                0.06,
                cam,
                transform=ax.transAxes,
                color="white",
                fontsize=8,
                bbox=dict(facecolor="black", alpha=0.55, edgecolor="none", pad=1.5),
            )

    def _draw_bev(self, frame: Dict[str, Any], ego_state: EgoState, ann: Annotations) -> None:
        ax = self.bev_ax
        ax.clear()
        ego_x = float(ego_state.pose.get("x", 0.0))
        ego_y = float(ego_state.pose.get("y", 0.0))
        _viz_configure_bev_ax(ax, ego_x, ego_y, self.bev_range_m)
        ax.set_title("BEV: Map, Actors, Ego Motion, Route, TL", fontsize=11)

        static_map_lookup: Optional[Dict[str, Dict[int, Any]]] = None
        if self.static_map is not None:
            static_map_lookup = _viz_draw_static_map(ax, self.static_map)

        route_line = build_route_line(frame.get("mission_goal"))
        if route_line is not None:
            rx, ry = route_line.xy
            ax.plot(
                rx,
                ry,
                color=TRAJECTORY_STYLES["route"].line_color,
                linewidth=TRAJECTORY_STYLES["route"].line_width,
                alpha=TRAJECTORY_STYLES["route"].line_alpha,
                linestyle=TRAJECTORY_STYLES["route"].line_style,
                zorder=TRAJECTORY_STYLES["route"].zorder,
            )

        _viz_draw_traffic_light_states(ax, ann, static_map_lookup)
        _viz_draw_objects(ax, ann, self.show_objects)
        _viz_add_trajectory(ax, getattr(ego_state, "trajectory_history", None), TRAJECTORY_STYLES["history"], trajectory_dt_s=self.trajectory_dt_s)
        _viz_add_trajectory(ax, getattr(ego_state, "trajectory_future", None), TRAJECTORY_STYLES["future"], trajectory_dt_s=self.trajectory_dt_s)
        _viz_draw_ego(ax, ego_state, _metadata_ego_dimensions(self.clip.metadata))
        _viz_add_legend(ax)

    def _load_lidar(self, path: str) -> Tuple[np.ndarray, Optional[np.ndarray]]:
        return load_pcd_points(path)

    def _draw_lidar(self, frame: Dict[str, Any]) -> int:
        ax = self.lidar_ax
        ax.clear()
        ax.set_aspect("equal")
        ax.set_facecolor("#111111")
        plot_range_m = self.lidar_range_m * 1.04
        ax.set_xlim(-plot_range_m, plot_range_m)
        ax.set_ylim(-plot_range_m, plot_range_m)
        ax.axis("off")
        ax.text(
            0.02,
            0.97,
            "Lidar top-down",
            transform=ax.transAxes,
            color="white",
            fontsize=9,
            ha="left",
            va="top",
            bbox=dict(facecolor="black", alpha=0.55, edgecolor="none", pad=1.5),
        )

        lidar_path = _frame_lidar_path(frame, self.clip.clip_path)
        if not lidar_path:
            return 0

        try:
            xyz, intensity = self._load_lidar(lidar_path)
        except Exception as exc:
            ax.text(0.5, 0.5, f"Lidar read failed\n{exc}", color="white", ha="center", va="center", transform=ax.transAxes, fontsize=8)
            return 0

        if xyz is None or xyz.size == 0:
            ax.text(0.5, 0.5, "No lidar points", color="white", ha="center", va="center", transform=ax.transAxes)
            return 0

        x, y, z = xyz[0, :], xyz[1, :], xyz[2, :]
        r = np.hypot(x, y)
        mask = (
            np.isfinite(xyz).all(axis=0)
            & (r >= 1.5)
            & (r <= self.lidar_range_m)
            & (z >= -2.5)
            & (z <= 4.0)
        )
        xyz = xyz[:, mask]
        if intensity is not None:
            intensity = intensity[mask]
        if xyz.size == 0:
            ax.text(0.5, 0.5, "No lidar points after filtering", color="white", ha="center", va="center", transform=ax.transAxes)
            return 0

        voxel_size_m = 0.5
        voxels = np.floor(xyz.T / voxel_size_m).astype(np.int32)
        _, unique_idx = np.unique(voxels, axis=0, return_index=True)
        xyz = xyz[:, unique_idx]
        if intensity is not None:
            intensity = intensity[unique_idx]

        if xyz.shape[1] > self.max_lidar_points:
            rng = np.random.default_rng(0)
            idx = rng.choice(xyz.shape[1], self.max_lidar_points, replace=False)
            xyz = xyz[:, idx]
            if intensity is not None:
                intensity = intensity[idx]

        colors: np.ndarray
        if intensity is not None and intensity.size:
            lo, hi = np.percentile(intensity, [2, 98])
            colors = np.clip(intensity, lo, hi) if hi > lo else intensity
        else:
            colors = xyz[2, :]

        ax.scatter(xyz[0, :], xyz[1, :], s=1.0, c=colors, cmap="viridis", alpha=0.75, linewidths=0)
        ax.scatter([0.0], [0.0], marker="*", s=80, c="#DE7061", edgecolors="white", linewidths=0.5)
        return int(xyz.shape[1])

    def _clear_axes_for_gc(self) -> None:
        for ax in list(self.cam_axes.values()):
            ax.clear()
        if self.center_cam_ax is not None:
            self.center_cam_ax.clear()
        self.bev_ax.clear()
        if self.lidar_ax is not None:
            self.lidar_ax.clear()

    def close(self) -> None:
        self.static_map = None
        self._clear_axes_for_gc()
        self.fig.clf()
        plt.close(self.fig)
        gc.collect()

    def render_frame(self, frame_idx: int) -> None:
        frame = self.clip.frames[frame_idx]
        ego_state = self._load_ego_state(frame)
        ann = self._load_annotations(frame)

        self._draw_cameras(frame, ego_state)
        self._draw_bev(frame, ego_state, ann)
        if self.lidar_ax is not None:
            self._draw_lidar(frame)

        self.fig.subplots_adjust(
            left=FIG_MARGIN,
            right=1.0 - FIG_MARGIN,
            bottom=FIG_MARGIN,
            top=1.0 - FIG_MARGIN,
            wspace=0,
            hspace=0,
        )

    def _save_static_frame(self, frame_idx: int, output_path: str, dpi: int) -> None:
        self.render_frame(frame_idx)
        os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
        self.fig.savefig(output_path, dpi=dpi, bbox_inches="tight", pad_inches=0, facecolor=self.fig.get_facecolor())

    def save_mp4(self, output_path: str, frame_indices: List[int], fps: int, dpi: int) -> None:
        os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
        self.fig.set_dpi(dpi)
        writer = FFMpegWriter(
            fps=max(fps, 1),
            codec="libx264",
            extra_args=["-pix_fmt", "yuv420p", "-movflags", "+faststart"],
        )
        with writer.saving(self.fig, output_path, dpi=dpi):
            for output_idx, frame_idx in enumerate(frame_indices, start=1):
                self.render_frame(frame_idx)
                writer.grab_frame(facecolor=self.fig.get_facecolor())
                self._clear_axes_for_gc()
                if output_idx % GC_COLLECT_EVERY_FRAMES == 0:
                    gc.collect()


def _select_frame_indices(total_frames: int, start_index: int, end_index: int, stride: int, max_frames: int) -> List[int]:
    if total_frames <= 0:
        return []
    start = max(0, start_index)
    end = total_frames - 1 if end_index < 0 else min(end_index, total_frames - 1)
    if end < start:
        return []
    indices = list(range(start, end + 1, max(1, stride)))
    if max_frames > 0:
        indices = indices[:max_frames]
    return indices


def _output_path_for_clip(output_dir: str, clip: ClipContext, as_image: bool) -> str:
    ext = "png" if as_image else "mp4"
    return os.path.join(output_dir, f"{clip.clip_name}.{ext}")


def _existing_playable_video(path: str) -> bool:
    if not os.path.isfile(path) or os.path.getsize(path) <= 0:
        return False
    ffprobe = shutil.which("ffprobe")
    if ffprobe is None:
        return True
    try:
        result = subprocess.run(
            [
                ffprobe,
                "-v",
                "error",
                "-select_streams",
                "v:0",
                "-show_entries",
                "stream=codec_type",
                "-of",
                "csv=p=0",
                path,
            ],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            check=False,
            timeout=10,
        )
    except (OSError, subprocess.TimeoutExpired):
        return False
    return result.returncode == 0 and "video" in result.stdout.lower()


def _existing_output_is_usable(path: str, as_image: bool) -> bool:
    if as_image:
        return os.path.isfile(path) and os.path.getsize(path) > 0
    return _existing_playable_video(path)


## Notebook Helpers

These helpers convert the configuration values into render options, render clips, and print a small dataset summary.

In [4]:
def build_render_options() -> Dict[str, Any]:
    return {
        "output_dir": OUTPUT_DIR,
        "start_index": START_INDEX,
        "end_index": END_INDEX,
        "stride": STRIDE,
        "max_frames": MAX_FRAMES,
        "fps": FPS,
        "dpi": DPI,
        "bev_range_m": BEV_RANGE_M,
        "render_lidar": RENDER_LIDAR,
        "lidar_range_m": LIDAR_RANGE_M,
        "max_lidar_points": MAX_LIDAR_POINTS,
        "trajectory_dt_s": TRAJECTORY_DT_S,
        "show_objects": SHOW_OBJECTS,
        "frame_idx": FRAME_IDX,
        "skip_existing": SKIP_EXISTING,
    }


def render_clip(clip: ClipContext, options: Dict[str, Any]) -> str:
    global SHOW_LIDAR_PANEL

    SHOW_LIDAR_PANEL = bool(options["render_lidar"])
    frame_idx = options["frame_idx"]
    as_image = frame_idx is not None
    out_path = _output_path_for_clip(options["output_dir"], clip, as_image=as_image)

    if as_image:
        if frame_idx < 0 or frame_idx >= len(clip.frames):
            return f"Skipping {clip.clip_name}: frame index {frame_idx} is out of range"
        frame_indices = [frame_idx]
    else:
        frame_indices = _select_frame_indices(
            len(clip.frames),
            options["start_index"],
            options["end_index"],
            options["stride"],
            options["max_frames"],
        )
        if not frame_indices:
            return f"Skipping {clip.clip_name}: no selected frames"

    if options["skip_existing"] and _existing_output_is_usable(out_path, as_image):
        output_kind = "output" if as_image else "playable output"
        return f"Skipping {clip.clip_name}: existing {output_kind} at {out_path}"

    viewer: Optional[NuReasoningDataViewer] = None
    try:
        viewer = NuReasoningDataViewer(
            clip,
            bev_range_m=options["bev_range_m"],
            lidar_range_m=options["lidar_range_m"],
            max_lidar_points=options["max_lidar_points"],
            trajectory_dt_s=options["trajectory_dt_s"],
            show_objects=options["show_objects"],
        )
        if frame_idx is not None:
            viewer._save_static_frame(frame_idx, out_path, options["dpi"])
            return f"Saved frame to {out_path}"

        viewer.save_mp4(out_path, frame_indices, fps=options["fps"], dpi=options["dpi"])
        return f"Saved MP4 to {out_path} ({len(frame_indices)} frames)"
    finally:
        if viewer is not None:
            viewer.close()


def render_clips(clips: List[ClipContext], options: Dict[str, Any]) -> List[str]:
    messages: List[str] = []
    for clip in clips:
        try:
            message = render_clip(clip, options)
        except Exception as exc:
            message = f"Failed {clip.clip_name}: {exc}"
        print(message)
        messages.append(message)
    return messages


def summarize_clip(clip: ClipContext) -> Dict[str, Any]:
    frame_count = len(clip.frames)
    camera_names = sorted(
        {
            camera_name
            for frame in clip.frames
            for camera_name in ((frame.get("sensors", {}) or {}).get("cameras", {}) or {}).keys()
        }
    )
    lidar_count = sum(1 for frame in clip.frames if _frame_lidar_path(frame, clip.clip_path))
    return {
        "clip_name": clip.clip_name,
        "clip_path": clip.clip_path,
        "frames": frame_count,
        "cameras": camera_names,
        "frames_with_lidar": lidar_count,
        "map_annotation": clip.metadata.get("map_annotation", ""),
        "metadata_keys": sorted(clip.metadata.keys()),
    }


def print_frame_overview(clip: ClipContext, frame_idx: int = 0) -> None:
    if frame_idx < 0 or frame_idx >= len(clip.frames):
        raise IndexError(f"frame_idx {frame_idx} out of range for {len(clip.frames)} frame(s)")
    frame = clip.frames[frame_idx]
    sensors = frame.get("sensors", {}) if isinstance(frame.get("sensors", {}), dict) else {}
    cameras = sensors.get("cameras", {}) if isinstance(sensors.get("cameras", {}), dict) else {}
    print(f"Clip: {clip.clip_name}")
    print(f"Frame index: {frame_idx}")
    print(f"Timestamp: {frame.get('timestamp_us', 'unknown')}")
    print(f"Ego state: {_resolve_path(str(frame.get('ego_state', '')), clip.clip_path)}")
    print(f"Annotations: {_resolve_path(str(frame.get('annotations', '')), clip.clip_path)}")
    print("Camera files:")
    for camera_name in CAMERA_ORDER:
        rel_path = cameras.get(camera_name, "")
        status = "found" if _safe_image(_resolve_path(str(rel_path), clip.clip_path)) is not None else "missing"
        print(f"  {camera_name:>11}: {status}  {rel_path}")
    lidar_path = _frame_lidar_path(frame, clip.clip_path)
    print(f"Lidar: {'found ' + lidar_path if lidar_path else 'missing'}")

## Discover Clips

This cell finds clip folders and reports the basic structure: frame count, camera names, lidar coverage, map annotation, and metadata keys.

In [5]:
clips = _discover_clips(INPUT_ROOT, CLIP_PATH, FILTER_TO_CLIPS_WITH_LIDAR, MAX_CLIPS)
if not clips:
    raise RuntimeError("No clip folders found with metadata.json and the selected filters.")

print(f"Discovered {len(clips)} clip(s)")
for clip in clips[:10]:
    summary = summarize_clip(clip)
    cameras = ", ".join(summary["cameras"]) or "none"
    print(f"- {summary['clip_name']}: {summary['frames']} frame(s), cameras: {cameras}, lidar frames: {summary['frames_with_lidar']}")
if len(clips) > 10:
    print(f"... {len(clips) - 10} more")

options = build_render_options()

Discovered 1 clip(s)
- 2023.07.31.23.53.58_KMHKM4AEXM1P98033_169ca87cd5ca5fdb912f770aa679edd7: 202 frame(s), cameras: back, back_left, back_right, front, front_left, front_right, left, right, lidar frames: 0


## Inspect One Frame

Use this before rendering to understand how one frame points to camera images, ego state, annotations, and optional lidar.

In [6]:
# Inspect the first selected clip and frame before rendering.
SELECTED_CLIP_INDEX = 0
SELECTED_FRAME_INDEX = 0

selected_clip = clips[SELECTED_CLIP_INDEX]
summary = summarize_clip(selected_clip)
print(summary)
print()
print_frame_overview(selected_clip, SELECTED_FRAME_INDEX)

{'clip_name': '2023.07.31.23.53.58_KMHKM4AEXM1P98033_169ca87cd5ca5fdb912f770aa679edd7', 'clip_path': '/home/etri/Jeongbin/VLA_nuReasoning_mini-project//home/etri/DATASET/nureasoning/clips/2023.07.31.23.53.58_KMHKM4AEXM1P98033_169ca87cd5ca5fdb912f770aa679edd7', 'frames': 202, 'cameras': ['back', 'back_left', 'back_right', 'front', 'front_left', 'front_right', 'left', 'right'], 'frames_with_lidar': 0, 'map_annotation': 'map.pkl', 'metadata_keys': ['camera_calibrations', 'clip_location', 'clip_token', 'ego_dimensions', 'end_timestamp_us', 'frame_rate_hz', 'frames', 'log_name', 'map_annotation', 'scenario_type', 'start_timestamp_us', 'total_frames']}

Clip: 2023.07.31.23.53.58_KMHKM4AEXM1P98033_169ca87cd5ca5fdb912f770aa679edd7
Frame index: 0
Timestamp: 1690849576699925
Ego state: /home/etri/Jeongbin/VLA_nuReasoning_mini-project//home/etri/DATASET/nureasoning/clips/2023.07.31.23.53.58_KMHKM4AEXM1P98033_169ca87cd5ca5fdb912f770aa679edd7/ego_state/1690849576699925.pkl
Annotations: /home/etri/J

## Render

If `RENDER_LIDAR = True` and a frame has no lidar file, the lidar panel remains empty for that frame.

In [7]:
# Render selected clips. For a quick test, set MAX_CLIPS=1 or FRAME_IDX=0 in the config cell.
messages = render_clips(clips, options)

Failed 2023.07.31.23.53.58_KMHKM4AEXM1P98033_169ca87cd5ca5fdb912f770aa679edd7: [Errno 2] No such file or directory: 'ffmpeg'


In [8]:
# Optional preview of the first rendered output.
from IPython.display import Image, Video, display

first_output = Path(_output_path_for_clip(OUTPUT_DIR, clips[0], as_image=FRAME_IDX is not None))
if not first_output.exists():
    print(f"No output to preview yet: {first_output}")
elif first_output.suffix.lower() == ".png":
    display(Image(filename=str(first_output)))
elif first_output.suffix.lower() == ".mp4":
    display(Video(filename=str(first_output), embed=True, html_attributes="controls"))
else:
    print(first_output)

No output to preview yet: nureasoning_viz/2023.07.31.23.53.58_KMHKM4AEXM1P98033_169ca87cd5ca5fdb912f770aa679edd7.mp4
